# Hypothesetest op de *mce* dataset

In dit notebook voer ik één hypothesetest uit op de dataset `mce.csv`. De dataset bevat speedrun-runs (o.a. tijd, platform en categorie).

**Onderzoeksvraag (voorbeeld):**
Zijn runs in de categorie **'Set Seed Any%'** op **PlayStation 4** gemiddeld sneller (lagere `speedrun_time`) dan op **PlayStation 3**?

> Let op: in deze dataset kan dezelfde speler meerdere runs hebben. Dat maakt observaties mogelijk niet volledig onafhankelijk. Voor dit leerarrangement behandelen we de runs hier als onafhankelijke metingen, maar noem dit wel als beperking in je reflectie.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pingouin as pg

# Pingouin gebruiken (handige statistiek-library). 
# Als pingouin nog niet geïnstalleerd is in jouw Deepnote environment, dan installeren we 'm automatisch.

# Data inlezen (Deepnote: meestal werkt 'mce.csv' direct)
df = pd.read_csv('mce.csv')

print(df.shape)
df.head()


## 1) Data selecteren en opschonen

We beperken ons tot:
- `cat_name == 'Set Seed Any%'`
- alleen platformen **PlayStation 4** en **PlayStation 3**
- `speedrun_time` als numerieke variabele (seconden).

In [ ]:
cat = 'Set Seed Any%'
platforms = ['PlayStation 4', 'PlayStation 3']

sub = df.loc[(df['cat_name'] == cat) & (df['platform_name'].isin(platforms)),
             ['run_id','player_id','platform_name','speedrun_time']].copy()
sub['speedrun_time'] = pd.to_numeric(sub['speedrun_time'], errors='coerce')
sub = sub.dropna(subset=['speedrun_time'])

sub['platform_name'].value_counts(), sub.shape

### Beschrijvende statistiek

In [ ]:
desc = sub.groupby('platform_name')['speedrun_time'].describe()
desc

### Visualisatie (boxplot)

Een boxplot helpt om scheefheid en uitschieters te zien.

In [ ]:
data_to_plot = [sub.loc[sub['platform_name']==p, 'speedrun_time'].values for p in platforms]

plt.figure(figsize=(7,4))
plt.boxplot(data_to_plot, labels=platforms, showmeans=True)
plt.ylabel('speedrun_time (seconden)')
plt.title(f"{cat}: speedrun_time per platform")
plt.show()

## 2) Hypotheses

We toetsen met significantieniveau **α = 0,05**.

- **H0 (nulhypothese):** de verdeling van `speedrun_time` is gelijk voor PS4 en PS3 (geen verschil).
- **H1 (alternatief, gericht):** PS4-runs zijn sneller: `speedrun_time(PS4) < speedrun_time(PS3)`.

Omdat de verdelingen duidelijk scheef kunnen zijn en de steekproef klein is, gebruiken we een **Mann–Whitney U-toets** (niet-parametrisch alternatief voor de onafhankelijke t-toets).

## 3) Aannames check (indicatief)

Voor een Mann–Whitney U-toets hoef je **geen normaliteit** aan te nemen, maar het is wel netjes om te laten zien hoe de verdeling eruitziet.

Hier gebruiken we **pingouin** om per groep een normaliteitstest te rapporteren (Shapiro-Wilk via `pg.normality`).

In [ ]:
ps4 = sub.loc[sub['platform_name']=='PlayStation 4', 'speedrun_time'].to_numpy()
ps3 = sub.loc[sub['platform_name']=='PlayStation 3', 'speedrun_time'].to_numpy()

print('n PS4:', len(ps4), ' | n PS3:', len(ps3))

# Normaliteit (Shapiro-Wilk) met pingouin
norm = pg.normality(data=sub, dv='speedrun_time', group='platform_name')
norm


## 4) Hypothesetoets uitvoeren (Mann–Whitney U)

We toetsen éénzijdig: **PS4 sneller dan PS3** ⇒ `alternative='less'` (want lagere tijd = sneller).

Met **pingouin** krijg je meteen ook effectmaten zoals:
- **RBC** = rank-biserial correlation
- **CLES** = common language effect size

In [ ]:
# Mann–Whitney U met pingouin
mwu_pg = pg.mwu(ps4, ps3, alternative='less')
mwu_pg


In [ ]:
alpha = 0.05
p = float(mwu_pg['p-val'].iloc[0])

if p < alpha:
    print(f"p = {p:.4g} < {alpha} → H0 verwerpen: PS4 is (statistisch) sneller dan PS3.")
else:
    print(f"p = {p:.4g} ≥ {alpha} → H0 niet verwerpen: geen statistisch bewijs dat PS4 sneller is dan PS3.")


## 5) Effectgrootte

Pingouin geeft al **RBC** en **CLES** terug. 

Hier berekenen we aanvullend:
- **Cliff's delta** (intuïtieve effectmaat; -1 = alles in PS4 lager dan PS3, +1 andersom)
- **Hodges–Lehmann** schatting (typisch verschil in seconden) + bootstrap 95%-BI

In [ ]:
# Effectmaten uit pingouin (uit de MWU-resultaten)
rbc = float(mwu_pg['RBC'].iloc[0]) if 'RBC' in mwu_pg.columns else None
cles = float(mwu_pg['CLES'].iloc[0]) if 'CLES' in mwu_pg.columns else None

def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    n1, n2 = len(x), len(y)
    greater = 0
    less = 0
    for xi in x:
        greater += np.sum(xi > y)
        less += np.sum(xi < y)
    return (greater - less) / (n1 * n2)

# Cliff's delta
cd = cliffs_delta(ps4, ps3)

# Hodges–Lehmann (median of all pairwise differences)
pairwise = np.subtract.outer(ps4, ps3).ravel()
hl = np.median(pairwise)

# Bootstrap 95% CI for Hodges–Lehmann
rng = np.random.default_rng(42)
B = 3000
boot = []
for _ in range(B):
    xb = rng.choice(ps4, size=len(ps4), replace=True)
    yb = rng.choice(ps3, size=len(ps3), replace=True)
    boot.append(np.median(np.subtract.outer(xb, yb).ravel()))
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

print("Pingouin RBC:", rbc)
print("Pingouin CLES:", cles)
print("Cliff's delta:", cd)
print("Hodges–Lehmann (PS4 - PS3) in sec:", hl)
print("Bootstrap 95% CI:", (ci_low, ci_high))


## 6) Conclusie (invullen op basis van output)

Schrijf hier in 4–6 regels:

- Welke hypothese je toetste (H0/H1)
- Welke toets je gebruikte en waarom (Mann–Whitney U; tijden zijn vaak scheef + kleine n)
- Wat de p-waarde was en je beslissing bij α = 0,05
- (Optioneel) wat de effectgrootte suggereert (RBC/CLES of Cliff’s delta + HL in seconden)